In [1]:
# ============================================================
# RAG USING 2 PDF FILES + CHROMADB + GEMINI
# NO LANGCHAIN
# GOOGLE COLAB - SINGLE CELL
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL LIBRARIES
# ------------------------------------------------------------

!pip install -q chromadb sentence-transformers google-genai pypdf



In [2]:
# ------------------------------------------------------------
# 2. IMPORT LIBRARIES
# ------------------------------------------------------------

import chromadb
from sentence_transformers import SentenceTransformer
from google import genai
from google.colab import files
from pypdf import PdfReader
import os

In [3]:
# ============================================================
# 3. GEMINI SETUP
# ============================================================

import os
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)
os.environ["GOOGLE_API_KEY"] = api_key
#import os
#from getpass import getpass

#os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API key: ")

In [6]:
# ============================================================
# 4. LOAD EMBEDDING MODEL
# ============================================================

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded.")


# ============================================================
# 5. UPLOAD EXACTLY 2 PDF FILES
# ============================================================

print()
print("Please upload TWO small PDF documents.")

uploaded = files.upload()

pdf_files = list(uploaded.keys())


if len(pdf_files) != 2:

    raise Exception(
        "Please upload exactly 2 PDF files."
    )


print()
print("Uploaded files:")

for f in pdf_files:

    print("-", f)


# ============================================================
# 6. FUNCTION TO EXTRACT TEXT FROM PDF
# ============================================================

def extract_pdf_text(pdf_file):

    reader = PdfReader(pdf_file)

    full_text = ""

    for page_number, page in enumerate(reader.pages):

        text = page.extract_text()

        if text:

            full_text += (
                f"\n\nPage {page_number + 1}\n"
                + text
            )

    return full_text

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.

Please upload TWO small PDF documents.


Saving 3088_Stock exchanges in India.pdf to 3088_Stock exchanges in India.pdf
Saving SENSEXCalculation.pdf to SENSEXCalculation.pdf

Uploaded files:
- 3088_Stock exchanges in India.pdf
- SENSEXCalculation.pdf


In [7]:

# ============================================================
# 7. READ BOTH PDF DOCUMENTS
# ============================================================

pdf_texts = {}


for pdf_file in pdf_files:

    print()
    print("Reading:", pdf_file)

    text = extract_pdf_text(
        pdf_file
    )

    pdf_texts[pdf_file] = text

    print(
        "Characters extracted:",
        len(text)
    )


# ============================================================
# 8. SPLIT TEXT INTO CHUNKS
# ============================================================

def chunk_text(
    text,
    chunk_size=800,
    overlap=150
):

    chunks = []

    start = 0


    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(
            chunk
        )

        start += (
            chunk_size
            -
            overlap
        )


    return chunks


# ============================================================
# 9. CREATE CHUNKS FROM BOTH PDFs
# ============================================================

documents = []

ids = []

metadatas = []


chunk_counter = 0


for pdf_file, text in pdf_texts.items():

    chunks = chunk_text(
        text
    )

    print(
        pdf_file,
        "->",
        len(chunks),
        "chunks"
    )


    for chunk_number, chunk in enumerate(chunks):

        if chunk.strip() == "":
            continue


        chunk_counter += 1


        documents.append(
            chunk
        )


        ids.append(
            f"chunk_{chunk_counter}"
        )


        metadatas.append({

            "source":
                pdf_file,

            "chunk":
                chunk_number

        })


print()
print(
    "Total chunks:",
    len(documents)
)



Reading: 3088_Stock exchanges in India.pdf
Characters extracted: 5493

Reading: SENSEXCalculation.pdf
Characters extracted: 6620
3088_Stock exchanges in India.pdf -> 9 chunks
SENSEXCalculation.pdf -> 11 chunks

Total chunks: 20


In [8]:


# ============================================================
# 10. CREATE EMBEDDINGS
# ============================================================

print()
print("Creating embeddings...")


document_embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
).tolist()


print(
    "Embeddings created."
)


# ============================================================
# 11. CREATE CHROMADB
# ============================================================

chroma_client = chromadb.Client()


# Delete old collection if it exists

try:

    chroma_client.delete_collection(
        "pdf_rag"
    )

except:

    pass


collection = chroma_client.create_collection(
    name="pdf_rag"
)


# ============================================================
# 12. STORE DOCUMENTS + EMBEDDINGS IN CHROMADB
# ============================================================

collection.add(

    ids=ids,

    documents=documents,

    embeddings=document_embeddings,

    metadatas=metadatas

)


print()
print(
    "Documents stored successfully in ChromaDB."
)


print(
    "Number of records:",
    collection.count()
)


# ============================================================
# 13. ASK USER A QUESTION
# ============================================================

print()
print(
    "=" * 60
)

question = input(
    "Ask a question about the two PDFs: "
)


# ============================================================
# 14. CREATE QUESTION EMBEDDING
# ============================================================

query_embedding = embedding_model.encode(
    question
).tolist()


# ============================================================
# 15. SEARCH CHROMADB
# ============================================================

results = collection.query(

    query_embeddings=[
        query_embedding
    ],

    n_results=4

)


# ============================================================
# 16. GET RETRIEVED CHUNKS
# ============================================================

retrieved_documents = (
    results["documents"][0]
)


retrieved_metadatas = (
    results["metadatas"][0]
)


retrieved_distances = (
    results["distances"][0]
)


# ============================================================
# 17. DISPLAY RETRIEVED CONTEXT
# ============================================================

print()
print(
    "=" * 60
)

print(
    "RETRIEVED CHUNKS"
)

print(
    "=" * 60
)


for i, (
    doc,
    metadata,
    distance
) in enumerate(

    zip(
        retrieved_documents,
        retrieved_metadatas,
        retrieved_distances
    ),

    start=1

):


    print()
    print(
        f"Result {i}"
    )

    print(
        "Source:",
        metadata["source"]
    )

    print(
        "Distance:",
        round(distance, 4)
    )

    print()

    print(
        doc[:500]
    )

    print(
        "..."
    )




Creating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings created.

Documents stored successfully in ChromaDB.
Number of records: 20

Ask a question about the two PDFs: What is BSE and what is Sensex?

RETRIEVED CHUNKS

Result 1
Source: SENSEXCalculation.pdf
Distance: 0.7606

loat methodology with  effect from September 1, 2003. Sensex is 
computed on free float market capitalization meth od because it is assumed that the shares held 
up by the promoters are locked and are not ava ilable for day to day trading. There are certain 
criteria's to determine the promoters holding and the company is required to update this 
information quarterly with the stock exchange. 
 
Let's now begin with the concept.         
        
Sensex is an abbreviation of the word Sensitive I
...

Result 2
Source: SENSEXCalculation.pdf
Distance: 1.0691

and Influence. It must 
also be noted that reliance industries is not permanently a part of index and may be removed 

Page 2
from the index if it fails the el igibility criteria. Any other company which ful

In [9]:

# ============================================================
# 18. BUILD CONTEXT FOR GEMINI
# ============================================================

context_parts = []


for doc, metadata in zip(

    retrieved_documents,

    retrieved_metadatas

):


    context_parts.append(

        f"""
SOURCE: {metadata['source']}

{doc}
"""

    )


context = "\n\n".join(
    context_parts
)


# ============================================================
# 19. CREATE RAG PROMPT
# ============================================================

prompt = f"""

You are a Retrieval-Augmented Generation assistant.

Answer the user's question using ONLY the
retrieved context from the PDF documents below.

If the answer is not present in the context,
say:

"I do not have enough information in the uploaded PDFs."

Do not invent information.

====================

RETRIEVED CONTEXT:

{context}

====================

USER QUESTION:

{question}

====================

ANSWER:

"""


# ============================================================
# 20. SEND CONTEXT + QUESTION TO GEMINI
# ============================================================

response = client.models.generate_content(

    model="gemini-2.5-flash",

    contents=prompt

)


# ============================================================
# 21. DISPLAY FINAL RAG ANSWER
# ============================================================

print()
print(
    "=" * 60
)

print(
    "FINAL RAG ANSWER"
)

print(
    "=" * 60
)

print()

print(
    response.text
)


FINAL RAG ANSWER

BSE stands for Bombay Stock Exchange.

Sensex is an abbreviation for Sensitive Index and is the benchmark index of the Bombay Stock Exchange (BSE). It is composed of 30 of the largest and most actively-traded stocks on the BSE. Initially compiled in 1986, it is the oldest stock index in India. The base year for Sensex is 1978-79 and its base value is 100. The Sensex was introduced to measure the ups and downs in the Indian stock market, with its movement reported based on price fluctuations in its 30 component companies.
